# CrisisSense-LLM — LoRA Fine-tuning (Qwen2.5-7B)


In [1]:
# STEP 1 — Install dependencies
!pip install -q torch transformers datasets peft accelerate sentencepiece

In [ ]:
# STEP 2 — Mount Google Drive (put your data/processed/ folder there)
from google.colab import drive
drive.mount('/content/drive')

In [3]:
# STEP 3 — Set paths
# Change this to wherever you uploaded your data in Google Drive
import os

DRIVE_ROOT = "/content/drive/MyDrive/disaster_llm"  # <-- change if needed

TRAIN_PATH = f"{DRIVE_ROOT}/train_v2.jsonl"
VAL_PATH   = f"{DRIVE_ROOT}/val_v2.jsonl"
OUTPUT_DIR = f"{DRIVE_ROOT}/outputs/qwen25_7b_lora"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Verify files exist
print("Train exists:", os.path.exists(TRAIN_PATH))
print("Val exists:  ", os.path.exists(VAL_PATH))

Train exists: True
Val exists:   True


In [4]:
MODEL_NAME  = "Qwen/Qwen2.5-7B-Instruct"
MAX_LENGTH  = 512          # reduced from 768
BATCH_SIZE  = 16           # bigger batch
GRAD_ACCUM  = 1            # no accumulation needed
LR          = 2e-5
EPOCHS      = 1            # early stop like the paper
WEIGHT_DECAY = 0.1
SAVE_EPOCH_FRACTION = 0.25  # save 4 checkpoints total
LORA_RANK   = 32
LORA_ALPHA  = 64
LORA_DROPOUT = 0.05

In [5]:
# STEP 5 — Load dataset
import json
from datasets import Dataset

def load_instruction_dataset(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            response_json = json.dumps(obj["response"], ensure_ascii=False)
            full_text = f"{obj['instruction']}\n{response_json}"
            records.append({"text": full_text, "instruction": obj["instruction"]})
    return Dataset.from_list(records)

train_dataset = load_instruction_dataset(TRAIN_PATH)
val_dataset   = load_instruction_dataset(VAL_PATH)
print(f"Train: {len(train_dataset):,} examples")
print(f"Val:   {len(val_dataset):,} examples")

Train: 513,400 examples
Val:   93,696 examples


In [6]:
# STEP 6 — Load tokenizer
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("Tokenizer loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded.


In [7]:
 # STEP 7 — Tokenize dataset
def make_tokenize_fn(tokenizer):
    def tokenize_fn(example):
        full = tokenizer(example["text"], truncation=True, max_length=MAX_LENGTH, padding=False)
        instr_ids = tokenizer(example["instruction"] + "\n", truncation=True, max_length=MAX_LENGTH, padding=False)["input_ids"]
        labels = full["input_ids"].copy()
        cutoff = min(len(instr_ids), len(labels))
        for i in range(cutoff):
            labels[i] = -100
        full["labels"] = labels
        return full
    return tokenize_fn

tokenize_fn = make_tokenize_fn(tokenizer)
train_dataset = train_dataset.map(tokenize_fn, remove_columns=["text", "instruction"])
val_dataset   = val_dataset.map(tokenize_fn,   remove_columns=["text", "instruction"])
print("Tokenization done.")

Map:   0%|          | 0/513400 [00:00<?, ? examples/s]

Map:   0%|          | 0/93696 [00:00<?, ? examples/s]

Tokenization done.


In [8]:
!pip install -q -U torchao


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 110.4 MB/s eta 0:00:00


In [9]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [10]:
# STEP 8 — Load model + apply LoRA
import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491


In [11]:
train_dataset = train_dataset.shuffle(seed=42).select(range(len(train_dataset) // 10))
val_dataset = val_dataset.shuffle(seed=42).select(range(len(val_dataset) // 10))
print(f"Train: {len(train_dataset):,}")
print(f"Val:   {len(val_dataset):,}")

Train: 51,340
Val:   9,369


In [12]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
print(f"Free memory: {torch.cuda.mem_get_info()[0]/1e9:.2f} GB")

Free memory: 26.30 GB


In [13]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.4 MB/s eta 0:00:00


In [14]:
# STEP 9 — Train (optimized for speed)
from transformers import Trainer, TrainingArguments
from transformers import DataCollatorForSeq2Seq

# Speed optimizations:
# 1. Bigger batch + less grad accum = fewer optimizer steps
# 2. tf32 on A100 = faster matrix math
# 3. dataloader workers = parallel data loading
# 4. group_by_length = less padding waste

BATCH_SIZE  = 4
GRAD_ACCUM  = 2

effective_batch_size = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = max(1, len(train_dataset) // effective_batch_size)
save_steps = max(1, int(steps_per_epoch * SAVE_EPOCH_FRACTION))
eval_steps = save_steps

print(f"Steps per epoch: {steps_per_epoch}")
print(f"Save/eval every: {save_steps} steps")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    bf16=True,
    tf32=True,
    logging_steps=20,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=10,
    eval_strategy="steps",
    eval_steps=eval_steps,
    report_to="none",
    remove_unused_columns=False,
    optim="adamw_torch_fused",
    gradient_checkpointing=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
    label_pad_token_id=-100,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Steps per epoch: 6417
Save/eval every: 1604 steps


Step,Training Loss,Validation Loss
1604,0.038709,0.044215
3208,0.039110,0.039365
4812,0.028827,0.034903


Step,Training Loss,Validation Loss
1604,0.038709,0.044215
3208,0.039110,0.039365
4812,0.028827,0.034903
6416,0.029550,0.034545
6418,0.029550,0.034544


TrainOutput(global_step=6418, training_loss=0.04421877815516131, metrics={'train_runtime': 8975.6756, 'train_samples_per_second': 5.72, 'train_steps_per_second': 0.715, 'total_flos': 7.43119847796179e+17, 'train_loss': 0.04421877815516131, 'epoch': 1.0})

In [15]:
# STEP 10 — Save final model
final_dir = f"{OUTPUT_DIR}/final"
trainer.save_model(final_dir)
tokenizer.save_pretrained(final_dir)
print(f"Model saved to {final_dir}")
print("Checkpoints saved to:", OUTPUT_DIR)

Model saved to /content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora/final
Checkpoints saved to: /content/drive/MyDrive/disaster_llm/outputs/qwen25_7b_lora
